In [1]:
import sys
#lets you access python's runtime environment
from pathlib import Path
#sys.path is a built in variable in the sys module and contains a list of directories that is seached through when you do an import
#so we are appending the src directory to that
sys.path.append(str(Path().resolve().parent / "src"))
import config

import pandas as pd
import numpy as np
import datetime

In [2]:
cleaned = pd.read_csv(config.PROJECT_ROOT/ "data" / "no_dupes.csv")
cleaned['Date'] = pd.to_datetime(cleaned['Date'], format = '%Y-%m-%d')

df_sorted = cleaned.sort_values(['Name', 'Year', 'Date'])

df_sorted['TimeCompetingYearEnd'] = pd.to_datetime(df_sorted['Year'].astype(str) + '-12-31') - df_sorted.groupby('Name')['Date'].transform('min')
df_sorted['TimeCompetingYearEnd'] = df_sorted['TimeCompetingYearEnd'].dt.days

df_sorted['NumberOfMeets'] = df_sorted.groupby(['Name', 'Year']).transform('size')

df_sorted['TimeCompeting'] = (
    df_sorted['Date'] - df_sorted.groupby('Name')['Date'].transform('min')
).dt.days

C:\Users\bnpar\AppData\Local\Temp\ipykernel_46744\1095282783.py:1: DtypeWarning: Columns (33,38) have mixed types. Specify dtype option on import or set low_memory=False.
  cleaned = pd.read_csv(config.PROJECT_ROOT/ "data" / "no_dupes.csv")


### Rate of improvement features

It might be expected that lifters that are making progress are more likely to continue competing. Therefore a set of features relating to the rate of progress are engineered.

In [4]:
df_sorted.columns

Index(['Name', 'Sex', 'Event', 'Equipment', 'Age', 'AgeClass',
       'BirthYearClass', 'Division', 'BodyweightKg', 'WeightClassKg',
       'Squat1Kg', 'Squat2Kg', 'Squat3Kg', 'Squat4Kg', 'Best3SquatKg',
       'Bench1Kg', 'Bench2Kg', 'Bench3Kg', 'Bench4Kg', 'Best3BenchKg',
       'Deadlift1Kg', 'Deadlift2Kg', 'Deadlift3Kg', 'Deadlift4Kg',
       'Best3DeadliftKg', 'TotalKg', 'Place', 'Dots', 'Wilks', 'Glossbrenner',
       'Goodlift', 'Tested', 'Country', 'State', 'Federation',
       'ParentFederation', 'Date', 'MeetCountry', 'MeetState', 'MeetTown',
       'MeetName', 'Sanctioned', 'Year', 'TimeCompetingYearEnd',
       'NumberOfMeets', 'TimeCompeting'],
      dtype='object')

In [5]:
#### USED TO COMPUTE (BEST MEET OF THIS YEAR - BEST MEET OF LAST YEAR) WHICH IS COMPUTED LATER AFTER TRANSFORMATION TO PANEL DATA

#Gives the best total of the year and the date that the time that lifter had been competing for when they achieved their best total
grouped = df_sorted.groupby(['Name', 'Year'])
df_sorted['BestTotalYear'] = grouped['TotalKg'].transform('max')
idx = grouped['TotalKg'].idxmax()
idx = idx.dropna()
time_best = df_sorted.loc[idx, ['Name', 'Year', 'TimeCompeting']]
time_best = time_best.rename(columns={'TimeCompeting': 'TimeCompetingBestTotalYear'})
df_sorted = df_sorted.merge(time_best, on = ['Name', 'Year'], how = 'left')

df_sorted['ImprovementGradientWithinYear'] = (
    (grouped['TotalKg'].transform('last') -
     grouped['TotalKg'].transform('first')) /
    (grouped['TimeCompeting'].transform('last') -
     grouped['TimeCompeting'].transform('first')).replace(0,np.nan)
)

df_sorted['PercentageImprovementGradientWithinYear'] = (
    df_sorted['ImprovementGradientWithinYear'] /
    (grouped['TotalKg'].transform('first')).replace(0,np.nan)
)



df_sorted['PrevMeetTotal'] = df_sorted.groupby('Name').shift(1)['TotalKg']
df_sorted['PrevMeetTimeCompeting'] = df_sorted.groupby('Name').shift(1)['TimeCompeting']

df_sorted['ImprovementGradientTwoMeets'] = (
    (df_sorted['TotalKg'] - df_sorted['PrevMeetTotal'])/
    (df_sorted['TimeCompeting']- df_sorted['PrevMeetTimeCompeting']).replace(0,np.nan)
)

df_sorted['PercentageImprovementGradientTwoMeets'] = df_sorted['ImprovementGradientTwoMeets']/df_sorted['PrevMeetTotal']




In [6]:
df_sorted['ImprovementAcceleration'] = (
    df_sorted.groupby('Name')['ImprovementGradientTwoMeets'].diff()/
    (df_sorted['TimeCompeting']- df_sorted['PrevMeetTimeCompeting']).replace(0,np.nan)
)

df_sorted['ImprovementAccelerationForPercentage'] = (
    df_sorted.groupby('Name')['PercentageImprovementGradientTwoMeets'].diff()/
    (df_sorted['TimeCompeting']- df_sorted['PrevMeetTimeCompeting']).replace(0,np.nan)
)

df_sorted = df_sorted.drop(columns = ['PrevMeetTotal', 'PrevMeetTimeCompeting'])

### Encoding federation 
trying federation by rank and federation by proportion


In [7]:
df_sorted['PrimaryFederation'] = (
    df_sorted.groupby(['Name', 'Year'])['Federation']
    .transform(lambda x: x.value_counts().index[0])
)

fed_counts = (
    df_sorted.groupby(['Year', 'Federation'])
    .size()
    .reset_index(name='FederationCount')
)

fed_counts['FederationRank'] = (
    fed_counts.groupby('Year')['FederationCount']
    .rank(method='first', ascending=False)
    .astype(int)
)

df_sorted = df_sorted.merge(
    fed_counts[['Year', 'Federation', 'FederationRank']],
    left_on=['Year', 'PrimaryFederation'],
    right_on = ['Year', 'Federation'],
    how='left'
)

df_sorted = df_sorted.drop(columns = 'Federation_y')
df_sorted = df_sorted.rename(columns = {'Federation_x': 'Federation'})
df_sorted['FederationRankCapped'] = df_sorted['FederationRank'].clip(upper=25)
df_sorted = df_sorted.drop(columns = 'FederationRank')

# Total number of competition entries per year
year_totals = (
    df_sorted.groupby('Year')
    .size()
    .reset_index(name='YearTotal')
)


fed_prop = fed_counts.merge(year_totals, on='Year', how='left')

fed_prop['FederationProportion'] = (
    fed_prop['FederationCount'] / fed_prop['YearTotal']
)

# Merge back
df_sorted = df_sorted.merge(
    fed_prop[['Year', 'Federation', 'FederationProportion', 'FederationCount']],
    left_on=['Year', 'PrimaryFederation'],
    right_on = ['Year', 'Federation'],
    how='left'
)

### Time since last pb

In [8]:
#Time since last PB at the end of the relevant year.

df_sorted['PB'] = df_sorted.groupby('Name')['TotalKg'].cummax()
df_sorted['PBBeforeMeet'] = df_sorted.groupby('Name')['PB'].shift(1)
df_sorted['IsPB'] = df_sorted['TotalKg']> df_sorted['PBBeforeMeet']
#first meet should count as PB
df_sorted['IsPB'] = df_sorted['PBBeforeMeet'].isna() | df_sorted['IsPB']
df_sorted['PBTimeCompeting'] = df_sorted['TimeCompeting'].where(df_sorted['IsPB'])
df_sorted['LastPBTimeCompeting'] = df_sorted.groupby('Name')['PBTimeCompeting'].ffill()
df_sorted['TimeSinceLastPBYearEnd'] = df_sorted['TimeCompetingYearEnd'] - df_sorted['LastPBTimeCompeting']

df_sorted = df_sorted.drop(columns = ['PB','PBBeforeMeet', 'IsPB', 'PBTimeCompeting', 'LastPBTimeCompeting'])

### Features related to number of attempts made per meet

In [9]:
bomb_squat = (
    ((df_sorted['Squat1Kg'] <= 0) | df_sorted['Squat1Kg'].isna()) &
    ((df_sorted['Squat2Kg'] <= 0) | df_sorted['Squat2Kg'].isna()) &
    ((df_sorted['Squat3Kg'] <= 0) | df_sorted['Squat3Kg'].isna()) & 
    ((df_sorted['Best3SquatKg'] <= 0) | df_sorted['Best3SquatKg'].isna())
)

bomb_bench = (
    ((df_sorted['Bench1Kg'] <= 0) | df_sorted['Bench1Kg'].isna()) &
    ((df_sorted['Bench2Kg'] <= 0) | df_sorted['Bench2Kg'].isna()) &
    ((df_sorted['Bench3Kg'] <= 0) | df_sorted['Bench3Kg'].isna()) & 
    ((df_sorted['Best3BenchKg'] <= 0) | df_sorted['Best3BenchKg'].isna())
)

bomb_deadlift = (
    ((df_sorted['Deadlift1Kg'] <= 0) | df_sorted['Deadlift1Kg'].isna()) &
    ((df_sorted['Deadlift2Kg'] <= 0) | df_sorted['Deadlift2Kg'].isna()) &
    ((df_sorted['Deadlift3Kg'] <= 0) | df_sorted['Deadlift3Kg'].isna()) & 
    ((df_sorted['Best3DeadliftKg'] <= 0) | df_sorted['Best3DeadliftKg'].isna())
)


df_sorted['BombOut'] = bomb_squat | bomb_bench | bomb_deadlift

Number of times a lifter has bombed out. <br>
Number of meets since last bombout, capped at 999. If lifter has never bombed out this is set to the maximum cap of 999.

In [10]:
df_sorted['NumberOfBombOuts'] = df_sorted.groupby('Name')['BombOut'].cumsum()
df_sorted['PercentageBombOuts'] = df_sorted['NumberOfBombOuts']/(df_sorted.groupby('Name').cumcount()+1)
df_sorted['MeetsSinceLastBombOut'] = df_sorted.groupby(['Name', 'NumberOfBombOuts']).cumcount()

#past capped number of bombouts it gets set to cap
df_sorted.loc[df_sorted['MeetsSinceLastBombOut']> config.CAP_MEETS_SINCE_BOMBOUT, 'MeetsSinceLastBombOut'] = config.CAP_MEETS_SINCE_BOMBOUT

#if lifter has never bombed out number of meets since last bombout is set to cap
df_sorted.loc[(df_sorted['NumberOfBombOuts'] ==0 ), 'MeetsSinceLastBombOut'] = config.CAP_MEETS_SINCE_BOMBOUT



Calculating the number of attempts the lifter made at the meet. If individual attempts weren't filled in but the lifter did not bomb out (e.g. only their best lift in each discipline was entered using the Best3SquatKg, Best3BenchKg, Best3DeadliftKg fields) then AttemptsMade is replaced with the mean number of attempts made by a lifter that did not bomb out.

In [11]:
attempt_cols = ['Squat1Kg','Squat2Kg','Squat3Kg',
                   'Bench1Kg','Bench2Kg','Bench3Kg',
                   'Deadlift1Kg','Deadlift2Kg','Deadlift3Kg']

df_sorted['AttemptsMade'] = (df_sorted[attempt_cols]>0).sum(axis=1)

#handling the case where the lifter didnt bombout but it's recorded as 0 attempts because individual attempst weren't filled in 
mask = (df_sorted['BombOut']==False)&(df_sorted['AttemptsMade']==0)

#excluding lifters who bombed out form calculation as we already know this lifter did not bomb out
mean_attempts = df_sorted.loc[(df_sorted['BombOut']==False) & (df_sorted['AttemptsMade']>0),'AttemptsMade'].mean().round() 

df_sorted.loc[mask, 'AttemptsMade'] = mean_attempts

Same value of below for all meets for that lifter for that year

In [12]:
df_sorted['LastMeetAttemptsMade'] = df_sorted.groupby(['Name', 'Year'])['AttemptsMade'].transform('last')
df_sorted['AverageAttemptsMade'] = df_sorted.groupby(['Name', 'Year'])['AttemptsMade'].transform('mean')
df_sorted['MostAttemptsMade'] = df_sorted.groupby(['Name', 'Year'])['AttemptsMade'].transform('max')
df_sorted['LeastAttemptsMade'] = df_sorted.groupby(['Name', 'Year'])['AttemptsMade'].transform('min')
df_sorted['BestMeetOfYear'] = df_sorted.groupby(['Name', 'Year'])['TotalKg'].transform('max')


In [13]:
# df_sorted['MeetsSinceLastBombOut'] = df_sorted.groupby(['Name', 'Year'])['MeetsSinceLastBombOut'].transform('last')
# df_sorted['NumberOfBombOuts'] = df_sorted.groupby(['Name', 'Year'])['NumberOfBombOuts'].transform('last')

print(len(df_sorted[df_sorted['AttemptsMade'] ==0])/len(df_sorted))

0.01054513102229467


In [14]:
df_sorted.columns

Index(['Name', 'Sex', 'Event', 'Equipment', 'Age', 'AgeClass',
       'BirthYearClass', 'Division', 'BodyweightKg', 'WeightClassKg',
       'Squat1Kg', 'Squat2Kg', 'Squat3Kg', 'Squat4Kg', 'Best3SquatKg',
       'Bench1Kg', 'Bench2Kg', 'Bench3Kg', 'Bench4Kg', 'Best3BenchKg',
       'Deadlift1Kg', 'Deadlift2Kg', 'Deadlift3Kg', 'Deadlift4Kg',
       'Best3DeadliftKg', 'TotalKg', 'Place', 'Dots', 'Wilks', 'Glossbrenner',
       'Goodlift', 'Tested', 'Country', 'State', 'Federation_x',
       'ParentFederation', 'Date', 'MeetCountry', 'MeetState', 'MeetTown',
       'MeetName', 'Sanctioned', 'Year', 'TimeCompetingYearEnd',
       'NumberOfMeets', 'TimeCompeting', 'BestTotalYear',
       'TimeCompetingBestTotalYear', 'ImprovementGradientWithinYear',
       'PercentageImprovementGradientWithinYear',
       'ImprovementGradientTwoMeets', 'PercentageImprovementGradientTwoMeets',
       'ImprovementAcceleration', 'ImprovementAccelerationForPercentage',
       'PrimaryFederation', 'FederationRank

### Features more easily computed from panel data

In [15]:
panel_data = df_sorted.groupby(['Name', 'Year']).tail(1).reset_index(drop = True)
panel_data = panel_data.sort_values(['Name', 'Year'])

panel_data['BestTotalPrevYear'] = panel_data.groupby('Name')['BestTotalYear'].shift(1)
panel_data['TimeCompetingBestTotalPrevYear'] = panel_data.groupby('Name')['TimeCompetingBestTotalYear'].shift(1)

panel_data['ImprovementGradientBetweenYears'] = (
    (panel_data['BestTotalYear'] - panel_data['BestTotalPrevYear'])/
    (panel_data['TimeCompetingBestTotalYear'] - panel_data['TimeCompetingBestTotalPrevYear']).replace(0, np.nan)
)
panel_data['PercentageImprovementGradientBetweenYears']= (
    panel_data['ImprovementGradientBetweenYears']/ (panel_data['BestTotalPrevYear']).replace(0, np.nan)
)

panel_data['MeetsYoYChange'] = panel_data.groupby('Name')['NumberOfMeets'].diff()

panel_data['AvgMeetsPerYear'] = (
    panel_data.groupby('Name')['NumberOfMeets']
    .expanding().mean()
    .reset_index(level=0, drop=True)
)
panel_data['BelowAvgMeets'] = (
    panel_data['NumberOfMeets'] < panel_data['AvgMeetsPerYear']
).astype(int)

In [16]:
cols =[
    'Name','Year', 'Date','WeightClassKg', 'Sex','Age', 'TimeCompetingYearEnd', 'NumberOfMeets',
    'Goodlift','BestMeetOfYear',  'AverageAttemptsMade', 'MostAttemptsMade',
    'LeastAttemptsMade','LastMeetAttemptsMade',
    'MeetsSinceLastBombOut', 'NumberOfBombOuts',
    'PercentageImprovementGradientWithinYear', 'ImprovementGradientWithinYear',
    'PercentageImprovementGradientTwoMeets', 'ImprovementGradientTwoMeets',
    'PercentageImprovementGradientBetweenYears', 'ImprovementGradientBetweenYears',
    'TimeSinceLastPBYearEnd',
    'PercentageBombOuts',
    'ImprovementAccelerationForPercentage', 'ImprovementAcceleration',
    'FederationRankCapped', 'FederationProportion', 'FederationCount',
    'MeetsYoYChange', 'AvgMeetsPerYear','BelowAvgMeets'
]
panel_data = panel_data.loc[:, cols]

#target column
panel_data['CompetesNextYear'] = (
    (panel_data['Name'] == panel_data['Name'].shift(-1)) & 
    (panel_data['Year'] +1 == panel_data['Year'].shift(-1))
)


#Need to drop the entries with the year as 2025 because we cannot know whether someone competed in 2026 yet.
max_year = panel_data['Year'].max()
panel_data = panel_data[panel_data['Year'] < max_year]

In [17]:
panel_data.columns

Index(['Name', 'Year', 'Date', 'WeightClassKg', 'Sex', 'Age',
       'TimeCompetingYearEnd', 'NumberOfMeets', 'Goodlift', 'BestMeetOfYear',
       'AverageAttemptsMade', 'MostAttemptsMade', 'LeastAttemptsMade',
       'LastMeetAttemptsMade', 'MeetsSinceLastBombOut', 'NumberOfBombOuts',
       'PercentageImprovementGradientWithinYear',
       'ImprovementGradientWithinYear',
       'PercentageImprovementGradientTwoMeets', 'ImprovementGradientTwoMeets',
       'PercentageImprovementGradientBetweenYears',
       'ImprovementGradientBetweenYears', 'TimeSinceLastPBYearEnd',
       'PercentageBombOuts', 'ImprovementAccelerationForPercentage',
       'ImprovementAcceleration', 'FederationRankCapped',
       'FederationProportion', 'FederationCount', 'MeetsYoYChange',
       'AvgMeetsPerYear', 'BelowAvgMeets', 'CompetesNextYear'],
      dtype='object')

### Encoding weight class and sex

Classes encoded as (weight class rank)/(number of classes) such that the heaviest weight class present in a given year for a given sex is 1 and the lightest weight class present in that year is 0.

Using HistGradientBoosting so can have categorical features 0,1,2 for F,M, Mx resepctively for Sex 

In [18]:
def w_cl_to_num(w):
    w = str(w)
    return float(w[:-1]) + 0.5 if w.endswith('+') else float(w)

panel_data['WeightClassKgNum'] = panel_data['WeightClassKg'].apply(w_cl_to_num)

panel_data['WeightClassKgCode'] = (
    panel_data.groupby(['Sex', 'Year'])['WeightClassKgNum']
    .transform(lambda x: (x.rank(method='dense')-1)/(x.nunique()-1))
)

panel_data = panel_data.drop(columns=['WeightClassKgNum'])

###one hot encoding used for Sex 
panel_data['Sex'] = panel_data['Sex'].map({'F': 0, 'M': 1, 'Mx': 2})


In [21]:
panel_data_path = config.PROJECT_ROOT/ "data" / "panel_data.csv"
panel_data.to_csv(panel_data_path, index = False)

In [20]:
panel_data.columns

Index(['Name', 'Year', 'Date', 'WeightClassKg', 'Sex', 'Age',
       'TimeCompetingYearEnd', 'NumberOfMeets', 'Goodlift', 'BestMeetOfYear',
       'AverageAttemptsMade', 'MostAttemptsMade', 'LeastAttemptsMade',
       'LastMeetAttemptsMade', 'MeetsSinceLastBombOut', 'NumberOfBombOuts',
       'PercentageImprovementGradientWithinYear',
       'ImprovementGradientWithinYear',
       'PercentageImprovementGradientTwoMeets', 'ImprovementGradientTwoMeets',
       'PercentageImprovementGradientBetweenYears',
       'ImprovementGradientBetweenYears', 'TimeSinceLastPBYearEnd',
       'PercentageBombOuts', 'ImprovementAccelerationForPercentage',
       'ImprovementAcceleration', 'FederationRankCapped',
       'FederationProportion', 'FederationCount', 'MeetsYoYChange',
       'AvgMeetsPerYear', 'BelowAvgMeets', 'CompetesNextYear',
       'WeightClassKgCode'],
      dtype='object')